In [ ]:
import sys
sys.path.append('..')

import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.datasets import mnist 
import my_cnn
from scipy.ndimage import rotate


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)
y_train = np.eye(10)[y_train]
y_test= np.eye(10)[y_test]

In [ ]:
angles = [-12, 0, 12]
new_x = []
new_y = []

for img, label in zip(x_train, y_train):
    for angle in angles:
        img_rot = rotate(img, angle, axes=(0,1), reshape=False, mode='constant', cval=0.0)
        new_x.append(img_rot)
        new_y.append(label)

x_train_augmented = np.array(new_x)
y_train_augmented = np.array(new_y)


In [ ]:
test_model = my_cnn.Model()
test_model.add(my_cnn.layers.Input((28, 28, 1)))
test_model.add(my_cnn.layers.Rescaling(1./255))
test_model.add(my_cnn.layers.Convolution2D(32, 3, activation_func='relu'))
test_model.add(my_cnn.layers.MaxPooling2D(2))
test_model.add(my_cnn.layers.Convolution2D(64, 3, activation_func='relu'))
test_model.add(my_cnn.layers.MaxPooling2D(2))
test_model.add(my_cnn.layers.Flatten())
test_model.add(my_cnn.layers.Dense(64, activation_func='relu'))
test_model.add(my_cnn.layers.Dense(10, activation_func='softmax'))

In [ ]:
test_model.compile(optimizer='momentum', loss='crossentropy', learning_rate=0.02)
test_model.fit(x_train_augmented, y_train_augmented, epochs=3, batch_size=128)

In [ ]:
model_loss, model_accuracy = test_model.evaluate(x_test, y_test)

In [ ]:
print(f"Loss: {model_loss}")
print(f"Accuracy: {model_accuracy*100:.2f}")

In [ ]:
predictions = test_model.predict(x_test)
decisions = np.argmax(predictions, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

print(classification_report(y_test_classes, decisions))


In [ ]:
cm = confusion_matrix(y_test_classes, decisions)
df_cm = pd.DataFrame(cm)
df_cm.index.name = "Real"
df_cm.columns.name = "Predicted"
df_cm 

In [ ]:
test_model.save('../models/custom_cnn.pkl')